In [1]:
import os
import re
import json
import sqlite3
from pathlib import Path
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect(r"text_simulation\simulation_output_multi_shot\deepseek-chat\deepseek-chat.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

cursor = conn.cursor()
cursor.execute("SELECT DISTINCT thread_id FROM checkpoints")
threads = cursor.fetchall()
print(len(threads))

31500


In [2]:
def process_response_text(pid, qid, response_text) -> dict:
    try:
        if "```json" in response_text:
            response_text_json = json.loads(response_text.split("```json")[1].split("```")[0])
        else:
            json_pattern = r'(\{.*\}|\[.*\])'
            match = re.search(json_pattern, response_text, re.DOTALL)
            response_text_json = json.loads(match.group(1))

    except (json.JSONDecodeError, IndexError) as e:
        print(f"Error parsing response JSON for: {e}")
        return {}
    return response_text_json

In [3]:
for pid in range(1, 501):
    response_json_to_all = {}
    for qid in range(1, 64):
        config = {"configurable": {"thread_id": f"pid_{pid}_Q{qid}"}}
        data = checkpointer.get(config)

        i = data["channel_values"]["messages"][-1].content.index('{')
        content = data["channel_values"]["messages"][-1].content[i:]
        response_json_to_one = process_response_text(pid, qid, content)
        response_json_to_all.update(response_json_to_one)
    with open(f"text_simulation/simulation_output_multi_shot/deepseek-chat/pid_{pid}_response.json", "w", encoding="utf-8") as f:
        final_json = {"persona_id": f"pid_{pid}", "response_text": json.dumps(response_json_to_all)}
        json.dump(final_json, f, ensure_ascii=False)

In [4]:
def calculate_qs_id(original_id):
    new_id = sorted(os.listdir(r"text_simulation\simulation_input_multi_shot\pid_1")).index(f"{original_id}.txt") + 1
    return new_id

In [5]:
calculate_qs_id('Q55')

51

In [6]:
config = {"configurable": {"thread_id": f"pid_109_Q55"}}
data = checkpointer.get(config)

content = data["channel_values"]["messages"][-1].content[-500:]
print(content)

mally favor free items) and his strong minimalist values (which oppose accumulating unnecessary possessions), I believe his minimalist values would prevail, especially since he's described as someone who "actively avoid[s] acquiring excess possessions" and whose "belongings are mindfully selected."

{
    "Q59": {
        "Question Type": "Single Choice",
        "Answers": {
            "SelectedByPosition": 2,
            "SelectedText": "No, I would not purchase the product"
        }
    }
}


In [7]:
s = """mally favor free items) and his strong minimalist values (which oppose accumulating unnecessary possessions), I believe his minimalist values would prevail, especially since he's described as someone who "actively avoid[s]"""
print(len(s))

222


In [8]:
conn.close()